# DDRI Bike-Change rep15 모델 비교

## 용어 설명

- `bike_change`(시간별 대여량 변화량): 현재 시간 `rental_count`에서 직전 시간 `rental_count`를 뺀 값
- `baseline`(기준선 모델): 추가 실험 전에 먼저 비교하는 기본 모델
- `RMSE`(제곱평균제곱근오차): 큰 오차에 더 민감한 대표 오차 지표
- `MAE`(평균절대오차): 오차 절대값의 평균
- `R²`(설명력): 모델이 타깃 변동을 얼마나 설명하는지 나타내는 지표
- `sign_accuracy`(부호 일치율): 예측값과 실제값의 증가/감소 방향이 같은 비율

목표:
- 대표 15개 `bike_change` 데이터셋에서 기본 회귀 모델들을 먼저 비교한다.
- 첫 라운드는 기존 방식 유지 원칙에 맞춰 `LightGBM regression`을 중심 기준선으로 삼되, `CatBoost`, `LinearRegression`, `Ridge`도 함께 본다.


In [1]:
from pathlib import Path

import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works/07_prediction_bike_change'
OUTPUT_DATA_DIR = WORK_DIR / 'output/data'

TRAIN_PATH = OUTPUT_DATA_DIR / 'ddri_prediction_bike_change_train_2023_2024.csv'
TEST_PATH = OUTPUT_DATA_DIR / 'ddri_prediction_bike_change_test_2025.csv'
METRICS_PATH = OUTPUT_DATA_DIR / 'ddri_bike_change_rep15_model_metrics.csv'


In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

FEATURE_COLUMNS = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h', 'rolling_mean_6h',
    'is_weekend', 'is_night_hour', 'is_commute_hour', 'hour_sin', 'is_rainy', 'hour_cos',
    'commute_morning_flag', 'commute_evening_flag', 'subway_distance_m', 'distance_naturepark_m',
    'restaurant_count_300m', 'convenience_store_count_300m', 'bus_stop_count_300m', 'cafe_count_300m',
    'elevation_diff_nearest_subway_m', 'nearest_park_area_sqm'
]
CATEGORICAL_COLUMNS = ['station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']

train_2023 = train_df[train_df['date'] < '2024-01-01'].copy()
valid_2024 = train_df[train_df['date'] >= '2024-01-01'].copy()
full_train = train_df.copy()

len(FEATURE_COLUMNS), len(CATEGORICAL_COLUMNS), train_2023.shape, valid_2024.shape, test_df.shape


(35, 7, (128880, 40), (131760, 40), (128880, 40))

In [3]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    sign_accuracy = ((pd.Series(y_pred) >= 0) == (y_true.reset_index(drop=True) >= 0)).mean()
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
        'sign_accuracy': round(float(sign_accuracy), 4),
    }

def prepare_tree_xy(df: pd.DataFrame):
    X = df[FEATURE_COLUMNS].copy()
    for col in CATEGORICAL_COLUMNS:
        X[col] = X[col].astype('category')
    y = df['bike_change'].copy()
    return X, y

def prepare_linear_xy(df: pd.DataFrame, dummy_columns=None):
    X = pd.get_dummies(df[FEATURE_COLUMNS], columns=CATEGORICAL_COLUMNS, drop_first=False)
    if dummy_columns is not None:
        X = X.reindex(columns=dummy_columns, fill_value=0)
    y = df['bike_change'].copy()
    return X, y


In [4]:
results = []

X_train_tree, y_train = prepare_tree_xy(train_2023)
X_valid_tree, y_valid = prepare_tree_xy(valid_2024)
X_full_tree, y_full = prepare_tree_xy(full_train)
X_test_tree, y_test = prepare_tree_xy(test_df)

X_train_linear, y_train_linear = prepare_linear_xy(train_2023)
dummy_columns = X_train_linear.columns.tolist()
X_valid_linear, y_valid_linear = prepare_linear_xy(valid_2024, dummy_columns)
X_full_linear, y_full_linear = prepare_linear_xy(full_train, dummy_columns)
X_test_linear, y_test_linear = prepare_linear_xy(test_df, dummy_columns)

lightgbm_rmse = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='regression',
)
lightgbm_rmse.fit(X_train_tree, y_train, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_RMSE', 'train_2023', y_train, lightgbm_rmse.predict(X_train_tree)))
results.append(evaluate_predictions('LightGBM_RMSE', 'validation_2024', y_valid, lightgbm_rmse.predict(X_valid_tree)))
lightgbm_rmse.fit(X_full_tree, y_full, categorical_feature=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('LightGBM_RMSE', 'train_full_refit', y_full, lightgbm_rmse.predict(X_full_tree)))
results.append(evaluate_predictions('LightGBM_RMSE', 'test_2025_refit', y_test, lightgbm_rmse.predict(X_test_tree)))

catboost_rmse = CatBoostRegressor(
    loss_function='RMSE',
    iterations=300,
    depth=8,
    learning_rate=0.05,
    random_seed=42,
    verbose=False,
)
catboost_rmse.fit(X_train_tree, y_train, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_RMSE', 'train_2023', y_train, catboost_rmse.predict(X_train_tree)))
results.append(evaluate_predictions('CatBoost_RMSE', 'validation_2024', y_valid, catboost_rmse.predict(X_valid_tree)))
catboost_rmse.fit(X_full_tree, y_full, cat_features=CATEGORICAL_COLUMNS)
results.append(evaluate_predictions('CatBoost_RMSE', 'train_full_refit', y_full, catboost_rmse.predict(X_full_tree)))
results.append(evaluate_predictions('CatBoost_RMSE', 'test_2025_refit', y_test, catboost_rmse.predict(X_test_tree)))

linear_model = LinearRegression()
linear_model.fit(X_train_linear, y_train_linear)
results.append(evaluate_predictions('LinearRegression', 'train_2023', y_train_linear, linear_model.predict(X_train_linear)))
results.append(evaluate_predictions('LinearRegression', 'validation_2024', y_valid_linear, linear_model.predict(X_valid_linear)))
linear_model.fit(X_full_linear, y_full_linear)
results.append(evaluate_predictions('LinearRegression', 'train_full_refit', y_full_linear, linear_model.predict(X_full_linear)))
results.append(evaluate_predictions('LinearRegression', 'test_2025_refit', y_test_linear, linear_model.predict(X_test_linear)))

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_linear, y_train_linear)
results.append(evaluate_predictions('Ridge', 'train_2023', y_train_linear, ridge_model.predict(X_train_linear)))
results.append(evaluate_predictions('Ridge', 'validation_2024', y_valid_linear, ridge_model.predict(X_valid_linear)))
ridge_model.fit(X_full_linear, y_full_linear)
results.append(evaluate_predictions('Ridge', 'train_full_refit', y_full_linear, ridge_model.predict(X_full_linear)))
results.append(evaluate_predictions('Ridge', 'test_2025_refit', y_test_linear, ridge_model.predict(X_test_linear)))

results_df = pd.DataFrame(results)
results_df.to_csv(METRICS_PATH, index=False, encoding='utf-8-sig')
print('saved:', METRICS_PATH)
results_df


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008433 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1956
[LightGBM] [Info] Number of data points in the train set: 128880, number of used features: 35
[LightGBM] [Info] Start training from score 0.000016


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011903 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1994
[LightGBM] [Info] Number of data points in the train set: 260640, number of used features: 35
[LightGBM] [Info] Start training from score -0.000004


/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=1.41491e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_bike_change_rep15_model_metrics.csv


/opt/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=6.99618e-18): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,model,split,rmse,mae,r2,sign_accuracy
0,LightGBM_RMSE,train_2023,0.9082,0.5852,0.6287,0.8988
1,LightGBM_RMSE,validation_2024,1.0055,0.6099,0.5414,0.8966
2,LightGBM_RMSE,train_full_refit,0.9272,0.5889,0.6115,0.8942
3,LightGBM_RMSE,test_2025_refit,0.8992,0.5482,0.5490,0.8908
4,CatBoost_RMSE,train_2023,0.9600,0.6009,0.5852,0.8803
5,CatBoost_RMSE,validation_2024,1.0079,0.6099,0.5392,0.8814
6,CatBoost_RMSE,train_full_refit,0.9592,0.5979,0.5842,0.8773
7,CatBoost_RMSE,test_2025_refit,0.9009,0.5489,0.5473,0.8792
8,LinearRegression,train_2023,1.1135,0.6844,0.4419,0.7450
9,LinearRegression,validation_2024,1.1070,0.6710,0.4442,0.7489


In [5]:
results_df[results_df['split'] == 'test_2025_refit'].sort_values('rmse').reset_index(drop=True)


,model,split,rmse,mae,r2,sign_accuracy
0,LightGBM_RMSE,test_2025_refit,0.8992,0.5482,0.5490,0.8908
1,CatBoost_RMSE,test_2025_refit,0.9009,0.5489,0.5473,0.8792
2,LinearRegression,test_2025_refit,0.9907,0.6066,0.4526,0.7408
3,Ridge,test_2025_refit,0.9907,0.6066,0.4526,0.7408
